In [ ]:
# =========================
# Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q laion-clap librosa soundfile pandas tqdm

import os
import re
import glob
import json
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
import laion_clap


In [ ]:
# ============================================
# CLAP evaluation using OFFICIAL laion-clap
# ============================================
# ============================================
# 1) PATHS
# ============================================
method_folders = {
    "raw_musicgen": "/content/drive/MyDrive/MusicGen_Project/Data/MusicGen_Output",
    "dsp": "/content/drive/MyDrive/MusicGen_Project/Data/DSP_Output",
    "flashsr": "/content/drive/MyDrive/MusicGen_Project/Data/FlashSR_Output",
    "audiosr": "/content/drive/MyDrive/MusicGen_Project/Data/AudioSR_Output",
    "apollo_from_audiosr": "/content/drive/MyDrive/MusicGen_Project/Data/Apollo_Output_From_AudioSR_To_96k",
}

output_root = "/content/drive/MyDrive/MusicGen_Project/Evaluation/CLAP_Results"
os.makedirs(output_root, exist_ok=True)

# ============================================
# 2) PROMPT MAP
# ============================================
prompt_map = {
    "piano_01": "a calm solo piano melody with soft emotional notes",
    "piano_02": "a bright classical piano piece with flowing arpeggios",
    "piano_03": "a melancholic piano composition with slow expressive playing",
    "piano_04": "an intimate jazz piano improvisation in a cozy atmosphere",
    "piano_05": "a cinematic grand piano theme with gentle dynamics",

    "guitar_01": "a warm acoustic guitar fingerpicking pattern with a natural sound",
    "guitar_02": "a soft classical guitar melody played with delicate touch",
    "guitar_03": "an energetic strummed acoustic guitar rhythm for a folk song",
    "guitar_04": "a bluesy electric guitar riff with smooth phrasing",
    "guitar_05": "a dreamy clean electric guitar melody with light reverb",

    "drums_01": "a tight acoustic drum groove with realistic kick and snare",
    "drums_02": "a punchy rock drum beat with strong snare hits",
    "drums_03": "a laid back jazz drum rhythm with brushes and swing feel",
    "drums_04": "a fast electronic drum pattern with crisp percussion",
    "drums_05": "a funky drum groove with syncopated rhythm and ghost notes",

    "ambient_01": "a peaceful ambient soundscape with soft pads and atmosphere",
    "ambient_02": "a dark ambient atmosphere with deep drones and texture",
    "ambient_03": "an ethereal ambient track with shimmering textures",
    "ambient_04": "a meditative ambient background with warm sustained sounds",
    "ambient_05": "a dreamy floating ambient composition with gentle motion",

    "electronic_01": "an upbeat electronic dance groove with synth bass and drums",
    "electronic_02": "a chill electronic track with soft synths and relaxed beat",
    "electronic_03": "a futuristic electronic piece with pulsing bass and synths",
    "electronic_04": "a synthwave instrumental with retro 1980s electronic sounds",
    "electronic_05": "a minimal electronic beat with deep bass and simple melody",

    "orchestral_01": "a cinematic orchestral theme with strings brass and percussion",
    "orchestral_02": "a dramatic orchestral soundtrack cue with rising intensity",
    "orchestral_03": "a soft orchestral piece led by strings and woodwinds",
    "orchestral_04": "an epic trailer style orchestra with powerful drums and brass",
    "orchestral_05": "a romantic orchestral melody with lush strings"
}

# ============================================
# 3) SETTINGS
# ============================================
TARGET_SR = 48000  # official CLAP repo uses 48 kHz audio
print("Loading official LAION-CLAP...")

# enable_fusion=False matches the standard non-fusion setup in repo examples
model = laion_clap.CLAP_Module(enable_fusion=False)
model.load_ckpt()   # downloads default pretrained checkpoint

print("Model loaded.")

# ============================================
# 4) HELPERS
# ============================================
def clean_id_from_filename(filename):
    name = os.path.splitext(os.path.basename(filename))[0]
    name = name.replace("_DSP", "")
    return name

def load_audio_for_clap(audio_path, target_sr=48000):
    audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    return audio.astype(np.float32)

def l2_normalize(x, axis=-1, eps=1e-12):
    norm = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.clip(norm, eps, None)

def compute_clap_similarity(audio_array, text):
    """
    audio_array: shape (T,)
    text: string
    returns cosine similarity
    """
    # laion-clap expects shape (N, T)
    audio_batch = np.expand_dims(audio_array, axis=0)

    audio_embed = model.get_audio_embedding_from_data(x=audio_batch, use_tensor=False)
    text_embed = model.get_text_embedding([text], use_tensor=False)

    audio_embed = l2_normalize(audio_embed)
    text_embed = l2_normalize(text_embed)

    score = float(np.sum(audio_embed * text_embed, axis=1)[0])
    return score

def evaluate_folder(folder_path, method_name, prompt_map):
    wav_files = sorted(glob.glob(os.path.join(folder_path, "*.wav")))
    print(f"\nMethod: {method_name}")
    print("Found files:", len(wav_files))

    rows = []

    for wav_path in tqdm(wav_files):
        file_name = os.path.basename(wav_path)
        base_id = clean_id_from_filename(file_name)

        if base_id not in prompt_map:
            print(f"[WARNING] Prompt not found for: {file_name} -> {base_id}")
            continue

        prompt = prompt_map[base_id]

        try:
            audio = load_audio_for_clap(wav_path, target_sr=TARGET_SR)
            clap_score = compute_clap_similarity(audio, prompt)

            rows.append({
                "method": method_name,
                "file": file_name,
                "base_id": base_id,
                "prompt": prompt,
                "duration_sec": len(audio) / TARGET_SR,
                "clap_score": clap_score
            })
        except Exception as e:
            print(f"[ERROR] {file_name}: {e}")

    df = pd.DataFrame(rows)

    per_file_csv = os.path.join(output_root, f"clap_scores_{method_name}.csv")
    df.to_csv(per_file_csv, index=False)
    print("Saved per-file CSV:", per_file_csv)

    if len(df) > 0:
        avg_score = df["clap_score"].mean()
        std_score = df["clap_score"].std()
    else:
        avg_score = np.nan
        std_score = np.nan

    summary = {
        "method": method_name,
        "num_files": len(df),
        "clap_score_mean": avg_score,
        "clap_score_std": std_score
    }

    return df, summary

# ============================================
# 5) TEST ONE FILE FIRST
# ============================================
test_file = "/content/drive/MyDrive/MusicGen_Project/Data/MusicGen_Output/ambient_01.wav"
test_prompt = prompt_map["ambient_01"]

audio = load_audio_for_clap(test_file, target_sr=TARGET_SR)
score = compute_clap_similarity(audio, test_prompt)
print("Test CLAP score:", score)

# ============================================
# 6) RUN ALL METHODS
# ============================================
all_dfs = []
summary_rows = []

for method_name, folder_path in method_folders.items():
    if not os.path.exists(folder_path):
        print(f"[SKIP] Folder does not exist: {folder_path}")
        continue

    df_method, summary = evaluate_folder(folder_path, method_name, prompt_map)

    if len(df_method) > 0:
        all_dfs.append(df_method)

    summary_rows.append(summary)

if len(all_dfs) > 0:
    all_files_df = pd.concat(all_dfs, ignore_index=True)
else:
    all_files_df = pd.DataFrame()

summary_df = pd.DataFrame(summary_rows)

all_files_csv = os.path.join(output_root, "clap_scores_all_files.csv")
summary_csv = os.path.join(output_root, "clap_scores_summary.csv")

all_files_df.to_csv(all_files_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print("\n===================================")
print("DONE")
print("Saved all-file CSV:", all_files_csv)
print("Saved summary CSV :", summary_csv)
print("===================================\n")

print(summary_df.sort_values("clap_score_mean", ascending=False))